In [1]:
import urllib.request
import ssl
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    ssl_context = ssl._create_unverified_context()

    with urllib.request.urlopen(url, context=ssl_context) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


File downloaded and saved as sms_spam_collection\SMSSpamCollection.tsv


In [3]:
import pandas as pd
df=pd.read_csv(data_file_path,sep="\t",header=None,names=["Label","Text"])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [4]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [9]:
def create_balanced_datasset(df):
    num_spam=df[df["Label"]=="spam"].shape[0]
    ham_subset=df[df["Label"]=="ham"].sample(num_spam,random_state=123)
    balanced_df=pd.concat([ham_subset,df[df["Label"]=="spam"]])
    return balanced_df
balanced_df=create_balanced_datasset(df)
balanced_df["Label"].value_counts()

Label
ham     747
spam    747
Name: count, dtype: int64

In [10]:
print(balanced_df.shape)

(1494, 2)


In [11]:
balanced_df["Label"]=balanced_df["Label"].map({"ham":0,"spam":1})

In [14]:
def random_split(df,train_part,val_part):
    df=df.sample(frac=1,random_state=123).reset_index(drop=True)
    train_end=int(len(df)*train_part)
    train_df=df[:train_end]
    val_end=train_end+int(len(df)*val_part)
    val_df=df[train_end:val_end]
    test_df=df[val_end:]
    return train_df,val_df,test_df
train_df,val_df,test_df=random_split(balanced_df,0.7,0.1)


In [15]:
train_df.to_csv("train.csv",index=None)
val_df.to_csv("val.csv",index=None)
test_df.to_csv("test.csv",index=None)

## Stage 2->Create Datalloader


In [16]:
import tiktoken

In [ ]:
#end of text for Padding

In [49]:
import torch
from torch.utils.data import DataLoader,Dataset
class SpamDataset(Dataset):
    def __init__(self,csv_file,tokenizer,max_length=None,pad_token_id=50256):
        self.data=pd.read_csv(csv_file)
        self.encoded_texts=[tokenizer.encode(text) for text in self.data["Text"]]
        if max_length is None:
            self.max_length=self.longest_encoded_length()
        else:
            self.max_length=max_length
            self.encoded_texts=[
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
                           ]
        self.encoded_texts=[
            encoded_text+[pad_token_id]*(self.max_length-len(encoded_text))
            for encoded_text in self.encoded_texts
                            ]
    def __getitem__(self,index):
            encoded=self.encoded_texts[index]
            label=self.data.iloc[index]["Label"]
            return (
                torch.tensor(encoded,dtype=torch.long),
                torch.tensor(label,dtype=torch.long)

            )
    def __len__(self):
            return len(self.data)
        
    def longest_encoded_length(self):
            max_length=0
            for encoded_text in self.encoded_texts:
                encoded_length=len(encoded_text)
                if encoded_length>max_length:
                    max_length=encoded_length
            return max_length



        

In [50]:
tokenizer=tiktoken.get_encoding("gpt2")

In [51]:
train_dataset=SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)
print(train_dataset.max_length)

120


In [52]:
val_dataset=SpamDataset(
    csv_file="val.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset=SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)


In [53]:
torch.manual_seed(123)
train_loader=DataLoader(
    dataset=train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    drop_last=True
)
val_loader=DataLoader(
    dataset=val_dataset,
    batch_size=8,
    num_workers=0,
    drop_last=False
)
test_loader=DataLoader(
    dataset=test_dataset,
    batch_size=8,
    num_workers=0,
    drop_last=False
)

In [55]:
print("Train")
for i,t in train_loader:
    print(i.shape)
    print(t.shape)
    break


Train
torch.Size([8, 120])
torch.Size([8])


In [56]:
print(f"{len(train_loader)} train \n {len(val_loader)} val\n {len(test_loader)} test")

130 train 
 19 val
 38 test


In [58]:
next(iter(train_loader))

[tensor([[11146, 50108,    25,   309,   742,    25, 42815,   284,  1400,    25,
            807,  3104,  3459,  1222,  1624,   534,  6721,   286,   513,  2250,
           1561,   640,   284,   779,   422,   534,  3072,   783,     0, 19808,
             21,  4579,    47,    14, 10295,   400,   753,   513,    71,  3808,
           1467,  2245,    30, 14116, 19485, 50256, 50256, 50256, 50256, 50256,
          50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
          50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
          50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
          50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
          50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
          50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
          50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
         [ 1639,   481,   307,  6464,  